In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from sqlalchemy import create_engine, text
import geopandas as gpd
from geopy.geocoders import Nominatim
from shapely.geometry import Point
import requests
import io

Reading in facility data for currently open hospitals as well as hospitals that have closed over the past 2 decades.

In [9]:
cms_providers = pd.read_csv('../data/Hospital_General_Information.csv')

In [10]:
cms_providers.head()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,NaN,15,15,NaN,10,10,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,NaN,15,15,NaN,10,10,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,NaN,15,15,NaN,10,9,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,3,2,NaN,15,5,NaN,10,7,NaN
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,1,5,2,29.0,15,10,29.0,10,7,29.0


In [11]:
cms_providers.columns

Index(['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State',
       'ZIP Code', 'County/Parish', 'Telephone Number', 'Hospital Type',
       'Hospital Ownership', 'Emergency Services',
       'Meets criteria for birthing friendly designation',
       'Hospital overall rating', 'Hospital overall rating footnote',
       'MORT Group Measure Count', 'Count of Facility MORT Measures',
       'Count of MORT Measures Better', 'Count of MORT Measures No Different',
       'Count of MORT Measures Worse', 'MORT Group Footnote',
       'Safety Group Measure Count', 'Count of Facility Safety Measures',
       'Count of Safety Measures Better',
       'Count of Safety Measures No Different',
       'Count of Safety Measures Worse', 'Safety Group Footnote',
       'READM Group Measure Count', 'Count of Facility READM Measures',
       'Count of READM Measures Better',
       'Count of READM Measures No Different', 'Count of READM Measures Worse',
       'READM Group Footnote', 'Pt Exp Gr

In [12]:
cms_providers['Hospital Type'].unique()

array(['Acute Care Hospitals', 'Acute Care - Veterans Administration',
       'Rural Emergency Hospital', 'Critical Access Hospitals',
       'Childrens', 'Psychiatric', 'Acute Care - Department of Defense',
       'Long-term'], dtype=object)

Narrowing down to just Acute Care Hospitals:

In [13]:
cms_providers = cms_providers[cms_providers['Hospital Type'] == 'Acute Care Hospitals']

Selecting only the columns that we need:

In [14]:
cms_providers = cms_providers[['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Emergency Services','Hospital Ownership']]

Renaming Facility ID to CCN for clarity:

In [15]:
cms_providers = cms_providers.rename(columns={'Facility ID': 'CCN', 'County/Parish': 'County', 'City/Town': 'City', 'ZIP Code': 'Zip'})


Adding a Closure column and setting to 0 since these hospitals are currently open.

In [16]:
 cms_providers['Closed'] = 0

In [17]:
cms_providers.head(2)

,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,Yes,Government - Hospital District or Authority,0
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,Yes,Government - Hospital District or Authority,0


Reading in rural hospital closures data:

In [18]:
rural_hospital_closures = pd.read_excel('../data/Closures-Database-for-Web.xlsx')

In [19]:
rural_hospital_closures.head()

,Hospital,Address,City,State,Zip,County/district,RUCA,CBSA,Medicare Payment,# of Beds,Closure Month,Closure Year,Complete Closure (0);\nConverted Closure (1),updated 5/18/2025
0,Bradford Regional Medical Center,116 INTERSTATE PARKWAY,BRADFORD,PA,16701.0,McKean,4.0,Micro,PPS,14.0,May,2026.0,1,196 Closed Rural Hospitals 2005-2026
1,Claremore Indian Hospital,101 South Moore Ave,Claremore,OK,74017.0,Rogers,4.0,Metro,IHS,46.0,October,2025.0,1,88 Converted to other health services
2,Glenn Medical Center,1133 W SYCAMORE ST,WILLOWS,CA,95988.0,Glenn,7.0,Neither,CAH,25.0,September,2025.0,1,NaN
3,Northern Light Inland Hospital,200 KENNEDY MEMORIAL DRIVE,WATERVILLE,ME,4901.0,Kennebec,4.0,Micro,MDH,33.0,May,2025.0,0,NaN
4,Lawrence Medical Center,202 HOSPITAL STREET,MOULTON,AL,35650.0,Lawrence,7.0,Metro,PPS,43.0,May,2025.0,1,NaN


In [20]:
rural_hospital_closures.columns

Index(['Hospital', 'Address', 'City', 'State', 'Zip', 'County/district',
       'RUCA', 'CBSA', 'Medicare Payment', '# of Beds', 'Closure Month',
       'Closure Year', 'Complete Closure (0);\nConverted Closure (1)',
       'updated 5/18/2025'],
      dtype='object')

Updating closure column so that complete closure = 2 and converted closure = 1:

In [21]:
rural_hospital_closures = rural_hospital_closures.rename(columns={'Complete Closure (0);\nConverted Closure (1)': 'Closed', 'County/district':'County', '# of Beds':'Number of Beds','Hospital':'Facility Name'})


In [22]:
rural_hospital_closures['Closed'] = rural_hospital_closures['Closed'].replace(0,2)

Remove footer:

In [23]:
rural_hospital_closures = rural_hospital_closures[rural_hospital_closures['Closed'] != '88 Converted to other health services']

Narrow down to just the columns we need and drop any fully null rows:

In [24]:
rural_hospital_closures = (rural_hospital_closures.drop(columns='updated 5/18/2025').dropna(how='all'))

Cast ZIP, number of beds, and closure year to integer:

In [25]:
rural_hospital_closures[['Zip','Number of Beds','Closure Year']] = rural_hospital_closures[['Zip','Number of Beds','Closure Year']].astype(int)


Convert closure month/year to date:

In [26]:
rural_hospital_closures['Closure Month'] = pd.to_datetime(rural_hospital_closures['Closure Month'], format='%B', errors='coerce').dt.month

In [27]:
rural_hospital_closures = rural_hospital_closures.rename(columns={'Closure Year':'year', 'Closure Month':'month'})

In [28]:
rural_hospital_closures['Closure Date'] = pd.to_datetime(rural_hospital_closures[['year', 'month']].assign(day=1))

In [29]:
rural_hospital_closures = rural_hospital_closures.drop(columns=['year','month'])

In [30]:
rural_hospital_closures.head(2)

,Facility Name,Address,City,State,Zip,County,RUCA,CBSA,Medicare Payment,Number of Beds,Closed,Closure Date
0,Bradford Regional Medical Center,116 INTERSTATE PARKWAY,BRADFORD,PA,16701,McKean,4.0,Micro,PPS,14,1,2026-05-01
1,Claremore Indian Hospital,101 South Moore Ave,Claremore,OK,74017,Rogers,4.0,Metro,IHS,46,1,2025-10-01


In [31]:
rural_hospital_closures['Closed'].value_counts()

Closed
2    108
1     88
Name: count, dtype: int64

Combine currently active hospital data with closed hospital data:

In [32]:
hospitals_master = pd.concat([cms_providers, rural_hospital_closures])

Uppercase all hospital names and addresses for consistency:

In [33]:
hospitals_master[['Facility Name','Address', 'City', 'State', 'County']] = (
                                                                                     hospitals_master[['Facility Name','Address', 'City', 'State', 'County']]
                                                                                     .apply(lambda x : x.str.upper())
                                                                                    )

In [34]:
hospitals_master

,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,RUCA,CBSA,Medicare Payment,Number of Beds,Closure Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,Yes,Government - Hospital District or Authority,0,NaN,NaN,NaN,NaN,NaT
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,Yes,Government - Hospital District or Authority,0,NaN,NaN,NaN,NaN,NaT
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,Yes,Proprietary,0,NaN,NaN,NaN,NaN,NaT
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,Yes,Voluntary non-profit - Private,0,NaN,NaN,NaN,NaN,NaT
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,Yes,Voluntary non-profit - Private,0,NaN,NaN,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,NaN,FAYETTE MEMORIAL HOSPITAL,543 N JACKSON,LA GRANGE,TX,78945,FAYETTE,NaN,NaN,2,7.0,Neither,PPS,24.0,2005-07-01
192,NaN,DELEON HOSPITAL,407 S TEXAS,DE LEON,TX,76444,COMANCHE,NaN,NaN,2,10.0,Neither,CAH,14.0,2005-07-01
193,NaN,MINNEWASKA DISTRICT HOSP,610 W 6TH ST PO BOX 160,STARBUCK,MN,56381,POPE,NaN,NaN,2,10.0,Neither,CAH,19.0,2005-06-01
194,NaN,GULF PINES HOSPITAL,102 20TH ST,PORT SAINT JOE,FL,32456,GULF,NaN,NaN,2,7.1,Metro,PPS,45.0,2005-03-01


In [35]:
def geocode_hospital_addresses(
    df: pd.DataFrame,
    address_col: str = "Address",
    city_col: str = "City",
    state_col: str = "State",
    zip_col: str = "Zip",
    facility_col: str = "Facility Name",
    user_agent: str = "hospital_cbsa_mapper",
    timeout: int = 10,
    min_delay_seconds: float = 1.0,
) -> pd.DataFrame:
    """
    Clean and geocode hospital addresses
    """
    df = hospitals_master.copy()
 
    # Work on string-safe copies
    df[address_col] = df[address_col].fillna("").astype(str)
    df[city_col] = df[city_col].fillna("").astype(str)
    df[state_col] = df[state_col].fillna("").astype(str)
    df[zip_col] = df[zip_col].fillna("").astype(str)
    df[facility_col] = df[facility_col].fillna("").astype(str)
 
    # Keep only the first comma-delimited segment (Some addresses already contain the city and state; we'll strip that off.)
    df[address_col] = df[address_col].str.split(",").str[0]
 
    # Remove suite / box / mail-slot style text
    df[address_col] = (
        df[address_col]
        .str.replace(
            r"\s*(?:SUITE|UNIT|STE|PO BOX|BOX|P O BOX|POST OFFICE BOX|MAIL SLOT)\s*\.?\s*\d+",
            "",
            case=False,
            regex=True,
        )
        .str.strip()
    )
 
    # Normalize highway phrasing like "123 U S HIGHWAY 45" -> "123 US-45"
    pattern = r"^(\d+)\s+(?:U\s*S\s+)?HIGHWAY\s+(\d+).*"
    df[address_col] = df[address_col].str.replace(pattern, r"\1 US-\2", regex=True)
 
    # Expand common written numbers / ordinals
    mapping = {
        "FIRST ": "1ST ",
        "SECOND ": "2ND ",
        "THIRD ": "3RD ",
        "FOURTH ": "4TH ",
        "FIFTH ": "5TH ",
        "SIXTH ": "6TH ",
        "SEVENTH ": "7TH ",
        "EIGHTH ": "8TH ",
        "NINTH ": "9TH ",
        "TENTH ": "10TH ",
        "ONE ": "1 ",
        "TWO ": "2 ",
        "THREE ": "3 ",
        "FOUR ": "4 ",
        "FIVE ": "5 ",
        "SIX ": "6 ",
        "SEVEN ": "7 ",
        "EIGHT ": "8 ",
        "NINE ": "9 ",
        "TEN ": "10 ",
        "TWNSHP PRKWY": "TOWNSHIP PARKWAY",
    }
 
    for word, replacement in mapping.items():
        df[address_col] = df[address_col].str.replace(word, replacement, regex=False)
 
    # Build full address
    df["Full Address"] = (
        df[address_col] + ", " +
        df[city_col] + ", " +
        df[state_col] + " " +
        df[zip_col]
    ).str.replace(r"\s+", " ", regex=True).str.strip(", ")
 
    # Initialize geocoder
    geolocator = Nominatim(user_agent=user_agent, timeout=timeout)
    geocode = RateLimiter(
        geolocator.geocode,
        min_delay_seconds=min_delay_seconds,
        swallow_exceptions=True,
    )
 
    # Primary geocoding
    df["Loc_Addr"] = df["Full Address"].apply(geocode)
    df["Loc_Name"] = df[facility_col].apply(geocode)
    df["Location"] = df["Loc_Addr"].combine_first(df["Loc_Name"])
 
    df["Latitude"] = df["Location"].apply(lambda loc: loc.latitude if loc else None)
    df["Longitude"] = df["Location"].apply(lambda loc: loc.longitude if loc else None)
 
    # Fallback: remove directional words and retry if still null
    direction_pattern = r"\b(NORTH|SOUTH|EAST|WEST|N|S|E|W|NO|SO)\b"
 
    df["Address_No_Direction"] = (
        df[address_col]
        .str.replace(direction_pattern, "", regex=True, case=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
 
    df["Full Address No Direction"] = (
        df["Address_No_Direction"] + ", " +
        df[city_col] + ", " +
        df[state_col] + " " +
        df[zip_col]
    ).str.replace(r"\s+", " ", regex=True).str.strip(", ")
 
    needs_retry = df["Location"].isna()
 
    df.loc[needs_retry, "Location_No_Direction"] = df.loc[needs_retry, "Full Address No Direction"].apply(geocode)
    df["Location_No_Direction"] = df["Location_No_Direction"].where(~df["Location"].isna(), df["Location_No_Direction"])
 
    # Fill missing primary location from fallback
    df["Location"] = df["Location"].combine_first(df["Location_No_Direction"])
 
    df["Latitude"] = df["Location"].apply(lambda loc: loc.latitude if loc else None)
    df["Longitude"] = df["Location"].apply(lambda loc: loc.longitude if loc else None)
 
    # Optional fallback by facility name if still missing
    still_missing = df["Location"].isna()
    if still_missing.any():
        df.loc[still_missing, "Location_Name_Fallback"] = df.loc[still_missing, facility_col].apply(geocode)
        df["Location"] = df["Location"].combine_first(df["Location_Name_Fallback"])
        df["Latitude"] = df["Location"].apply(lambda loc: loc.latitude if loc else None)
        df["Longitude"] = df["Location"].apply(lambda loc: loc.longitude if loc else None)
        
    # Drop everything except lat and long
    #df = df.drop(columns=['Loc_Addr','Loc_Name','Location','Address_No_Direction','Location_No_Direction','Latitude_No_Direction','Longitude_No_Direction'])
 
    return df

In [36]:
hospitals_master = geocode_hospital_addresses(hospitals_master)

hospitals_master.head()

,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,...,Full Address,Loc_Addr,Loc_Name,Location,Latitude,Longitude,Address_No_Direction,Full Address No Direction,Location_No_Direction,Location_Name_Fallback
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,Yes,Government - Hospital District or Authority,0,...,"1108 ROSS CLARK CIRCLE, DOTHAN, AL 36301","(1108, Ross Clark Circle, Dothan, Houston Coun...","(Hillsboro Medical Center, 335, Southeast 8th ...","(1108, Ross Clark Circle, Dothan, Houston Coun...",31.214885,-85.361261,1108 ROSS CLARK CIRCLE,"1108 ROSS CLARK CIRCLE, DOTHAN, AL 36301",NaN,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 US-431,BOAZ,AL,35957,MARSHALL,Yes,Government - Hospital District or Authority,0,...,"2505 US-431, BOAZ, AL 35957","(Marshall Medical Center South, 2505, Florida ...",None,"(Marshall Medical Center South, 2505, Florida ...",34.221402,-86.160960,2505 US-431,"2505 US-431, BOAZ, AL 35957",NaN,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,Yes,Proprietary,0,...,"1701 VETERANS DRIVE, FLORENCE, AL 35630","(North Alabama Medical Center, 1701, Veterans ...","(North Alabama Medical Center, 1701, Veterans ...","(North Alabama Medical Center, 1701, Veterans ...",34.805043,-87.650814,1701 VETERANS DRIVE,"1701 VETERANS DRIVE, FLORENCE, AL 35630",NaN,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,Yes,Voluntary non-profit - Private,0,...,"702 N MAIN ST, OPP, AL 36467","(Mizell Memorial Hospital, 702, North Main Str...","(Mizell Memorial Hospital, 702, North Main Str...","(Mizell Memorial Hospital, 702, North Main Str...",31.292879,-86.254900,702 MAIN ST,"702 MAIN ST, OPP, AL 36467",NaN,NaN
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,Yes,Voluntary non-profit - Private,0,...,"50 MEDICAL PARK EAST DRIVE, BIRMINGHAM, AL 35235",None,"(St. Vincent's East, 50, Medical Park Drive Ea...","(St. Vincent's East, 50, Medical Park Drive Ea...",33.595315,-86.667631,50 MEDICAL PARK DRIVE,"50 MEDICAL PARK DRIVE, BIRMINGHAM, AL 35235",NaN,NaN


In [42]:
hospitals_master['Latitude'].isna().sum()

np.int64(107)

In [40]:
hospitals_master['Coordinates'] = hospitals_master['Latitude'].astype(str) + ', ' + hospitals_master['Longitude'].astype(str)

In [43]:
hospitals_master[['Facility Name','Address','Coordinates']].to_csv('../data/nominatim_geocoded.csv', index=False)